# 🧬 Dark Manifold Virtual Cell

## A Quantum Field Theory Approach to Whole-Cell Simulation

This notebook implements the **Dark Manifold** architecture for simulating JCVI-syn3A, the world's simplest self-replicating cell (493 genes).

### Key Features:
- **Quantum Field Representation**: Molecular species as field excitations
- **Dark Matter Interaction Field**: Learned mediator replacing explicit rate constants
- **Perturbation Prediction**: Predicts knockout effects without seeing them

### Current Model: 100 genes covering:
- Glycolysis (12 genes)
- Pentose Phosphate Pathway (7 genes)
- Nucleotide Synthesis (15 genes)
- Amino Acid tRNA Synthetases (20 genes)
- Lipid Synthesis (10 genes)
- ATP Synthase Complex (10 genes)
- Core Transcription/Translation (15 genes)
- Transporters (13 genes)

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q torch numpy matplotlib

# Clone the repository
!git clone https://github.com/YOUR_USERNAME/dark-manifold-cell.git 2>/dev/null || echo "Already cloned"
%cd dark-manifold-cell

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Define JCVI-syn3A Pathways (100 genes)

In [ ]:
# JCVI-syn3A pathway definitions
PATHWAYS = {
    'glycolysis': {
        'genes': ['ptsI', 'ptsH', 'pgi', 'pfkA', 'fba', 'tpiA', 'gapA', 'pgk', 'gpmI', 'eno', 'pyk', 'ldh'],
        'metabolites': ['Glc_ext', 'Glc', 'G6P', 'F6P', 'FBP', 'DHAP', 'GAP', 'BPG', 'PG3', 'PG2', 'PEP', 'Pyr', 'Lac'],
    },
    'pentose_phosphate': {
        'genes': ['zwf', 'pgl', 'gnd', 'rpe', 'rpiA', 'tktA', 'talA'],
        'metabolites': ['G6P', 'PGL', 'PG6', 'Ru5P', 'X5P', 'R5P', 'S7P', 'E4P'],
    },
    'nucleotide_synthesis': {
        'genes': ['purA', 'purB', 'purC', 'purD', 'purE', 'purF', 'purH', 'purK', 'purL', 'purM', 'purN', 'pyrB', 'pyrC', 'pyrD', 'pyrE'],
        'metabolites': ['PRPP', 'IMP', 'AMP', 'GMP', 'UMP', 'CMP', 'dATP', 'dGTP', 'dCTP', 'dTTP'],
    },
    'amino_acid': {
        'genes': ['alaS', 'argS', 'asnS', 'aspS', 'cysS', 'glnS', 'gluS', 'glyS', 'hisS', 'ileS',
                  'leuS', 'lysS', 'metS', 'pheS', 'proS', 'serS', 'thrS', 'trpS', 'tyrS', 'valS'],
        'metabolites': ['Ala', 'Arg', 'Asn', 'Asp', 'Cys', 'Gln', 'Glu', 'Gly', 'His', 'Ile',
                       'Leu', 'Lys', 'Met', 'Phe', 'Pro', 'Ser', 'Thr', 'Trp', 'Tyr', 'Val'],
    },
    'lipid_synthesis': {
        'genes': ['accA', 'accB', 'accC', 'accD', 'fabD', 'fabF', 'fabG', 'fabH', 'fabI', 'fabZ'],
        'metabolites': ['AcCoA', 'MalCoA', 'ACP', 'FA_C16', 'FA_C18', 'PG', 'CL'],
    },
    'energy': {
        'genes': ['atpA', 'atpB', 'atpC', 'atpD', 'atpE', 'atpF', 'atpG', 'atpH', 'ndk', 'adk'],
        'metabolites': ['ATP', 'ADP', 'AMP', 'GTP', 'GDP', 'NAD', 'NADH', 'NADP', 'NADPH'],
    },
    'transcription_translation': {
        'genes': ['rpoA', 'rpoB', 'rpoC', 'rpoD', 'rpsA', 'rpsB', 'rplA', 'rplB', 'fusA', 'tufA',
                  'tsf', 'infA', 'infB', 'infC', 'prfA'],
        'metabolites': ['NTP_pool', 'AA_pool', 'tRNA_charged', 'mRNA', 'Protein'],
    },
    'transport': {
        'genes': ['glcU', 'nupC', 'potA', 'potB', 'potC', 'potD', 'oppA', 'oppB', 'oppC', 'oppD',
                  'oppF', 'secA', 'secY'],
        'metabolites': ['Glc_ext', 'Nuc_ext', 'Spd_ext', 'AA_ext', 'Peptide_ext'],
    },
}

# Build gene and metabolite lists
GENES = []
GENE_TO_PATHWAY = {}
for pathway, data in PATHWAYS.items():
    for gene in data['genes']:
        if gene not in GENES:
            GENES.append(gene)
            GENE_TO_PATHWAY[gene] = pathway

METABOLITES = []
for pathway, data in PATHWAYS.items():
    for met in data['metabolites']:
        if met not in METABOLITES:
            METABOLITES.append(met)

print(f"Total genes: {len(GENES)}")
print(f"Total metabolites: {len(METABOLITES)}")
print(f"\nPathways:")
for p, d in PATHWAYS.items():
    print(f"  {p}: {len(d['genes'])} genes")

## 3. Ground Truth Model (Michaelis-Menten Kinetics)

In [ ]:
class Syn3A100GeneModel:
    """Ground truth metabolic model based on JCVI-syn3A."""

    def __init__(self):
        self.n_genes = len(GENES)
        self.n_mets = len(METABOLITES)
        self.genes = GENES
        self.metabolites = METABOLITES

        np.random.seed(42)

        # Build stoichiometry matrix
        self.S = self._build_stoichiometry()
        self.Vmax = np.random.uniform(0.5, 2.0, self.n_genes)
        self.Km = np.random.uniform(0.1, 1.0, self.n_genes)
        self.W_reg = self._build_regulation()
        self.expression = np.random.uniform(0.5, 2.0, self.n_genes)

    def _build_stoichiometry(self):
        S = np.zeros((self.n_mets, self.n_genes))

        # Glycolysis pathway
        glyc_genes = PATHWAYS['glycolysis']['genes']
        glyc_mets = PATHWAYS['glycolysis']['metabolites']

        for i, gene in enumerate(glyc_genes):
            if gene in GENES:
                g_idx = GENES.index(gene)
                if i < len(glyc_mets) - 1:
                    sub_idx = METABOLITES.index(glyc_mets[i]) if glyc_mets[i] in METABOLITES else -1
                    prod_idx = METABOLITES.index(glyc_mets[i+1]) if glyc_mets[i+1] in METABOLITES else -1
                    if sub_idx >= 0:
                        S[sub_idx, g_idx] = -1
                    if prod_idx >= 0:
                        S[prod_idx, g_idx] = 1

        # ATP production
        atp_idx = METABOLITES.index('ATP') if 'ATP' in METABOLITES else -1
        adp_idx = METABOLITES.index('ADP') if 'ADP' in METABOLITES else -1

        if atp_idx >= 0 and adp_idx >= 0:
            for gene in ['pgk', 'pyk']:
                if gene in GENES:
                    g_idx = GENES.index(gene)
                    S[atp_idx, g_idx] = 1
                    S[adp_idx, g_idx] = -1

            if 'pfkA' in GENES:
                g_idx = GENES.index('pfkA')
                S[atp_idx, g_idx] = -1
                S[adp_idx, g_idx] = 1

        # Energy genes
        for gene in PATHWAYS['energy']['genes']:
            if gene in GENES and 'atp' in gene.lower():
                g_idx = GENES.index(gene)
                if atp_idx >= 0:
                    S[atp_idx, g_idx] = 0.5
                if adp_idx >= 0:
                    S[adp_idx, g_idx] = -0.5

        # Other pathways
        for pathway, data in PATHWAYS.items():
            if pathway in ['glycolysis', 'energy']:
                continue
            for gene in data['genes']:
                if gene in GENES:
                    g_idx = GENES.index(gene)
                    for met in data['metabolites'][:3]:
                        if met in METABOLITES:
                            m_idx = METABOLITES.index(met)
                            S[m_idx, g_idx] = np.random.choice([-1, 1]) * np.random.uniform(0.2, 0.8)

        return S

    def _build_regulation(self):
        W = np.zeros((self.n_genes, self.n_genes))
        for i in range(self.n_genes):
            n_reg = np.random.randint(2, 6)
            regulators = np.random.choice(self.n_genes, n_reg, replace=False)
            for r in regulators:
                if r != i:
                    W[i, r] = np.random.choice([-1, 1]) * np.random.uniform(0.1, 0.5)
        return W

    def dynamics(self, state, enzyme_mask=None):
        if enzyme_mask is None:
            enzyme_mask = np.ones(self.n_genes)

        enzymes = state[:self.n_genes] * enzyme_mask
        mets = state[self.n_genes:]
        dstate = np.zeros_like(state)

        # Enzyme dynamics
        reg_effect = np.tanh(self.W_reg @ enzymes)
        dstate[:self.n_genes] = 0.01 * (self.expression * (1 + 0.5 * reg_effect) - enzymes)

        # Metabolite dynamics
        for g in range(self.n_genes):
            substrates = np.where(self.S[:, g] < 0)[0]
            products = np.where(self.S[:, g] > 0)[0]

            if len(substrates) > 0:
                S_conc = np.mean([mets[s] for s in substrates])
                rate = self.Vmax[g] * enzymes[g] * S_conc / (self.Km[g] + S_conc + 1e-6)

                for s in substrates:
                    dstate[self.n_genes + s] -= rate * abs(self.S[s, g])
                for p in products:
                    dstate[self.n_genes + p] += rate * abs(self.S[p, g])

        # ATP consumption
        atp_idx = METABOLITES.index('ATP') if 'ATP' in METABOLITES else -1
        if atp_idx >= 0:
            dstate[self.n_genes + atp_idx] -= 0.1 * mets[atp_idx]

        return dstate

    def simulate(self, initial, n_steps=100, enzyme_mask=None, dt=0.05):
        traj = [initial.copy()]
        state = initial.copy()

        for _ in range(n_steps):
            dstate = self.dynamics(state, enzyme_mask)
            state = state + dt * dstate
            state = np.clip(state, 0.01, 50)
            traj.append(state.copy())

        return np.array(traj)

    def get_initial_state(self):
        state = np.zeros(self.n_genes + self.n_mets)
        state[:self.n_genes] = self.expression
        state[self.n_genes:] = np.random.uniform(0.5, 2.0, self.n_mets)

        atp_idx = METABOLITES.index('ATP') if 'ATP' in METABOLITES else -1
        adp_idx = METABOLITES.index('ADP') if 'ADP' in METABOLITES else -1
        if atp_idx >= 0:
            state[self.n_genes + atp_idx] = 3.0
        if adp_idx >= 0:
            state[self.n_genes + adp_idx] = 1.0

        return state

# Initialize ground truth
gt = Syn3A100GeneModel()
print(f"Ground truth model: {gt.n_genes} genes, {gt.n_mets} metabolites")

## 4. Dark Manifold Architecture

In [ ]:
class DarkManifold100Gene(nn.Module):
    """Dark Manifold architecture for 100-gene syn3A model."""

    def __init__(self, n_genes, n_mets, hidden_dim=256):
        super().__init__()
        self.n_genes = n_genes
        self.n_mets = n_mets
        self.total_dim = n_genes + n_mets

        # Dark field encoder
        self.dark_encoder = nn.Sequential(
            nn.Linear(self.total_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
        )

        # Learned stoichiometry
        self.W_stoich = nn.Parameter(torch.zeros(n_mets, n_genes))

        # Gene regulatory network
        self.W_reg = nn.Parameter(torch.zeros(n_genes, n_genes))

        # Metabolite dynamics
        self.met_dynamics = nn.Sequential(
            nn.Linear(hidden_dim + n_mets, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, n_mets),
        )

        # Gene expression dynamics
        self.gene_dynamics = nn.Sequential(
            nn.Linear(hidden_dim + n_genes, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, n_genes),
        )

    def forward(self, state, enzyme_mask=None):
        batch = state.shape[0]

        if enzyme_mask is None:
            enzyme_mask = torch.ones(batch, self.n_genes, device=state.device)

        enzymes = state[:, :self.n_genes] * enzyme_mask
        mets = state[:, self.n_genes:]

        # Dark field encoding
        masked_state = torch.cat([enzymes, mets], dim=-1)
        dark = self.dark_encoder(masked_state)

        # Enzyme-mediated changes
        W_s = torch.tanh(self.W_stoich) * 0.5
        stoich_effect = (W_s @ enzymes.T).T

        # Metabolite dynamics
        met_input = torch.cat([dark, mets], dim=-1)
        dmets = self.met_dynamics(met_input)
        dmets = dmets + stoich_effect * mets

        # Gene dynamics
        W_r = torch.tanh(self.W_reg) * 0.3
        reg_effect = (W_r @ enzymes.T).T
        gene_input = torch.cat([dark, enzymes], dim=-1)
        dgenes = self.gene_dynamics(gene_input)
        dgenes = 0.1 * (dgenes + reg_effect)

        # Update
        new_enzymes = enzymes + 0.01 * dgenes
        new_enzymes = F.relu(new_enzymes) + 0.01
        new_enzymes = new_enzymes * enzyme_mask

        new_mets = mets + 0.02 * dmets
        new_mets = F.relu(new_mets) + 0.01

        return torch.cat([new_enzymes, new_mets], dim=-1)

    def simulate(self, initial, n_steps, enzyme_mask=None):
        trajectory = [initial]
        state = initial

        for _ in range(n_steps):
            state = self.forward(state, enzyme_mask)
            trajectory.append(state)

        return torch.stack(trajectory)

# Initialize model
model = DarkManifold100Gene(gt.n_genes, gt.n_mets).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Generate Training Data

In [ ]:
print("Generating training data...")

data = []
initial = gt.get_initial_state()

# Wild-type trajectories
print("  Wild-type trajectories...")
for _ in range(100):
    init = initial * (1 + 0.2 * np.random.randn(len(initial)))
    init = np.clip(init, 0.1, 20)
    traj = gt.simulate(init, n_steps=50)
    mask = np.ones(gt.n_genes)

    for i in range(len(traj) - 1):
        data.append((traj[i], traj[i+1], mask))

# Knockout trajectories
ko_genes = list(range(0, gt.n_genes, 5))  # Every 5th gene
print(f"  Knockout trajectories ({len(ko_genes)} genes)...")

for ko in ko_genes:
    mask = np.ones(gt.n_genes)
    mask[ko] = 0

    for _ in range(10):
        init = initial.copy()
        init[:gt.n_genes] *= mask
        init = init * (1 + 0.1 * np.random.randn(len(init)))
        init = np.clip(init, 0.1, 20)

        traj = gt.simulate(init, n_steps=50, enzyme_mask=mask)
        for i in range(len(traj) - 1):
            data.append((traj[i], traj[i+1], mask))

print(f"\nTotal samples: {len(data)}")

# Convert to tensors
inputs = torch.tensor(np.array([d[0] for d in data]), dtype=torch.float32).to(device)
targets = torch.tensor(np.array([d[1] for d in data]), dtype=torch.float32).to(device)
masks = torch.tensor(np.array([d[2] for d in data]), dtype=torch.float32).to(device)

## 6. Training

In [ ]:
# Training parameters
n_epochs = 500
batch_size = 128
lr = 1e-3

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs)

history = {'loss': []}

print("Training...")
for epoch in range(n_epochs):
    model.train()

    # Random batch
    idx = torch.randint(0, len(inputs), (batch_size,))
    batch_in = inputs[idx]
    batch_target = targets[idx]
    batch_mask = masks[idx]

    # Forward
    pred = model(batch_in, batch_mask)
    loss = F.mse_loss(pred, batch_target)

    # Sparsity regularization
    loss = loss + 0.0001 * (torch.abs(model.W_stoich).mean() +
                            torch.abs(model.W_reg).mean())

    # Backward
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()

    history['loss'].append(loss.item())

    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1:4d}: Loss = {loss.item():.6f}")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(history['loss'])
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.yscale('log')
plt.grid(True)
plt.show()

## 7. Evaluation

In [ ]:
model.eval()

print("=" * 60)
print("EVALUATION")
print("=" * 60)

# Trajectory prediction
print("\n1. TRAJECTORY PREDICTION")
initial = gt.get_initial_state()
gt_traj = gt.simulate(initial, n_steps=100)

with torch.no_grad():
    model_traj = model.simulate(
        torch.tensor(initial, dtype=torch.float32).unsqueeze(0).to(device),
        n_steps=100
    ).cpu().squeeze(1).numpy()

corrs = []
for t in [25, 50, 75, 100]:
    gt_mets = gt_traj[t, gt.n_genes:]
    model_mets = model_traj[t, gt.n_genes:]
    corr = np.corrcoef(gt_mets, model_mets)[0, 1]
    corrs.append(corr)
    print(f"   t={t}: {corr:.4f}")

avg_traj = np.mean(corrs)
print(f"   Average: {avg_traj:.4f}")

# Knockout predictions
print("\n2. KNOCKOUT PREDICTIONS")
test_genes = ['ptsI', 'pfkA', 'pyk', 'atpA', 'rpoB', 'fusA', 'accA', 'purF']
test_indices = [GENES.index(g) for g in test_genes if g in GENES]

atp_idx = METABOLITES.index('ATP') if 'ATP' in METABOLITES else 0
ko_results = []

for ko_idx in test_indices:
    gene_name = GENES[ko_idx]

    with torch.no_grad():
        # Wild-type
        wt_traj = gt.simulate(gt.get_initial_state(), n_steps=100)
        wt_atp = wt_traj[-1, gt.n_genes + atp_idx]

        # Knockout
        mask = np.ones(gt.n_genes)
        mask[ko_idx] = 0
        ko_init = gt.get_initial_state()
        ko_init[:gt.n_genes] *= mask
        ko_traj = gt.simulate(ko_init, n_steps=100, enzyme_mask=mask)
        ko_atp = ko_traj[-1, gt.n_genes + atp_idx]

        gt_delta = ko_atp - wt_atp

        # Model predictions
        model_wt = model.simulate(
            torch.tensor(gt.get_initial_state(), dtype=torch.float32).unsqueeze(0).to(device),
            n_steps=100
        )[-1, 0, gt.n_genes + atp_idx].cpu().item()

        model_ko = model.simulate(
            torch.tensor(ko_init, dtype=torch.float32).unsqueeze(0).to(device),
            100,
            torch.tensor(mask, dtype=torch.float32).unsqueeze(0).to(device)
        )[-1, 0, gt.n_genes + atp_idx].cpu().item()

        model_delta = model_ko - model_wt

        ko_results.append({
            'gene': gene_name,
            'gt_atp': gt_delta,
            'model_atp': model_delta,
        })

        sign_match = '✓' if np.sign(gt_delta) == np.sign(model_delta) else '✗'
        print(f"   {gene_name:6s}: GT={gt_delta:+.2f}, Model={model_delta:+.2f} {sign_match}")

# Summary
gt_atps = np.array([r['gt_atp'] for r in ko_results])
model_atps = np.array([r['model_atp'] for r in ko_results])
atp_corr = np.corrcoef(gt_atps, model_atps)[0, 1] if np.std(gt_atps) > 0.01 else 0
sign_correct = sum(1 for r in ko_results if np.sign(r['gt_atp']) == np.sign(r['model_atp']))

print("\n" + "=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)
print(f"""
╔════════════════════════════════════════════════════════════════════╗
║  JCVI-syn3A 100-GENE MODEL                                         ║
╠════════════════════════════════════════════════════════════════════╣
║  Trajectory Correlation:     {avg_traj:.3f}                                ║
║  ATP Prediction Correlation: {atp_corr:.3f}                                ║
║  Sign Accuracy:              {sign_correct}/{len(ko_results)} ({100*sign_correct/len(ko_results):.0f}%)                              ║
╚════════════════════════════════════════════════════════════════════╝
""")

if avg_traj > 0.9 and sign_correct >= 6:
    print("🟢 SUCCESS: Dark Manifold learns 100-gene syn3A metabolism!")
elif avg_traj > 0.7:
    print("🟡 PROMISING: Good trajectory prediction")
else:
    print("🔴 NEEDS MORE TRAINING")

## 8. Visualize Results

In [ ]:
# Plot trajectories
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# ATP trajectory
atp_idx = METABOLITES.index('ATP')
axes[0, 0].plot(gt_traj[:, gt.n_genes + atp_idx], 'b-', label='Ground Truth')
axes[0, 0].plot(model_traj[:, gt.n_genes + atp_idx], 'r--', label='Dark Manifold')
axes[0, 0].set_xlabel('Time Step')
axes[0, 0].set_ylabel('ATP Concentration')
axes[0, 0].set_title('ATP Dynamics')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Pyruvate trajectory
pyr_idx = METABOLITES.index('Pyr')
axes[0, 1].plot(gt_traj[:, gt.n_genes + pyr_idx], 'b-', label='Ground Truth')
axes[0, 1].plot(model_traj[:, gt.n_genes + pyr_idx], 'r--', label='Dark Manifold')
axes[0, 1].set_xlabel('Time Step')
axes[0, 1].set_ylabel('Pyruvate Concentration')
axes[0, 1].set_title('Pyruvate Dynamics')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Knockout effects
genes = [r['gene'] for r in ko_results]
x = np.arange(len(genes))
width = 0.35

axes[1, 0].bar(x - width/2, gt_atps, width, label='Ground Truth', color='blue', alpha=0.7)
axes[1, 0].bar(x + width/2, model_atps, width, label='Dark Manifold', color='red', alpha=0.7)
axes[1, 0].set_xlabel('Gene')
axes[1, 0].set_ylabel('ATP Change')
axes[1, 0].set_title('Knockout Effects on ATP')
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(genes, rotation=45)
axes[1, 0].legend()
axes[1, 0].grid(True, axis='y')
axes[1, 0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)

# Learned stoichiometry
W_stoich = torch.tanh(model.W_stoich).detach().cpu().numpy()
im = axes[1, 1].imshow(W_stoich[:20, :20], cmap='RdBu', vmin=-0.5, vmax=0.5)
axes[1, 1].set_xlabel('Gene Index')
axes[1, 1].set_ylabel('Metabolite Index')
axes[1, 1].set_title('Learned Stoichiometry (partial)')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.savefig('dark_manifold_results.png', dpi=150)
plt.show()

## 9. Save Model

In [ ]:
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'genes': GENES,
    'metabolites': METABOLITES,
    'results': {
        'trajectory_corr': avg_traj,
        'atp_corr': atp_corr,
        'sign_accuracy': sign_correct / len(ko_results),
    },
}, 'dark_manifold_100gene.pt')

print("✓ Model saved to dark_manifold_100gene.pt")

# Download model
from google.colab import files
files.download('dark_manifold_100gene.pt')